# Week 3: RAG Pipeline — M&A Due Diligence
Multi-stage retrieval pipeline: `RecursiveCharacterTextSplitter → all-MiniLM-L6-v2 → ChromaDB → CrossEncoder → Gemini`

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

from src.ingestion.pdf_loader import load_pdf
from src.ingestion.chunker import chunk_document
from src.ingestion.embedder import embed_chunks, embed_query
from src.tools.vector_store import retrieve, collection_stats
from src.tools.reranker import rerank
from src.prompts.rag_prompts import rag_query_prompt
from src.llm import gemini_client

print('Imports successful')

## Step 1: Load, Chunk, and Embed PDFs

In [ ]:
# This was already run via the pipeline script
# Inspect ChromaDB stats
stats = collection_stats(persist_dir='../outputs/chromadb')
print(stats)

## Step 2: Retrieve + Rerank + Generate

### Query: What are the most significant litigation risks and legal contingencies for each company?

In [ ]:
query = "What are the most significant litigation risks and legal contingencies for each company?"
q_emb = embed_query(query)
companies = ["Apple", "Microsoft", "Nvidia"]
all_top_chunks = []
for company in companies:
    candidates = retrieve(q_emb, top_k=10, persist_dir='../outputs/chromadb', where={"company": company})
    top_comp = rerank(query, candidates, top_k=2)
    all_top_chunks.extend(top_comp)

all_top_chunks.sort(key=lambda x: x['rerank_score'], reverse=True)
print(f'Final Chunks after per-company reranking:')
for i, c in enumerate(all_top_chunks, 1):
    print(f"  [{i}] {c['metadata']['company']} | {c['metadata']['section']} | score={c['rerank_score']:.4f}")

In [ ]:
prompt = rag_query_prompt(query, all_top_chunks)
# Uncomment to call Gemini
# response = gemini_client.query(prompt)
# print(response)

### Query: Compare the unrecognized tax benefit positions across Apple, Microsoft, and Nvidia.

In [ ]:
query = "Compare the unrecognized tax benefit positions across Apple, Microsoft, and Nvidia."
q_emb = embed_query(query)
companies = ["Apple", "Microsoft", "Nvidia"]
all_top_chunks = []
for company in companies:
    candidates = retrieve(q_emb, top_k=10, persist_dir='../outputs/chromadb', where={"company": company})
    top_comp = rerank(query, candidates, top_k=2)
    all_top_chunks.extend(top_comp)

all_top_chunks.sort(key=lambda x: x['rerank_score'], reverse=True)
print(f'Final Chunks after per-company reranking:')
for i, c in enumerate(all_top_chunks, 1):
    print(f"  [{i}] {c['metadata']['company']} | {c['metadata']['section']} | score={c['rerank_score']:.4f}")

In [ ]:
prompt = rag_query_prompt(query, all_top_chunks)
# Uncomment to call Gemini
# response = gemini_client.query(prompt)
# print(response)

### Query: What are the largest contractual purchase obligations and how do they compare?

In [ ]:
query = "What are the largest contractual purchase obligations and how do they compare?"
q_emb = embed_query(query)
companies = ["Apple", "Microsoft", "Nvidia"]
all_top_chunks = []
for company in companies:
    candidates = retrieve(q_emb, top_k=10, persist_dir='../outputs/chromadb', where={"company": company})
    top_comp = rerank(query, candidates, top_k=2)
    all_top_chunks.extend(top_comp)

all_top_chunks.sort(key=lambda x: x['rerank_score'], reverse=True)
print(f'Final Chunks after per-company reranking:')
for i, c in enumerate(all_top_chunks, 1):
    print(f"  [{i}] {c['metadata']['company']} | {c['metadata']['section']} | score={c['rerank_score']:.4f}")

In [ ]:
prompt = rag_query_prompt(query, all_top_chunks)
# Uncomment to call Gemini
# response = gemini_client.query(prompt)
# print(response)

## Saved Results
All results (JSON + markdown report) saved to `outputs/responses/` and `outputs/reports/`.